In [1]:
"""
Simple Linear Regression: TV advertising vs Sales  (ISLR Advertising dataset)
=============================================================================
Model:  Sales = b0 + b1 * TV + error

Run:  python tv_sales_linear_regression.py
Needs: pandas, numpy, matplotlib, seaborn, scipy, scikit-learn, statsmodels
"""


'\nSimple Linear Regression: TV advertising vs Sales  (ISLR Advertising dataset)\n=============================================================================\nModel:  Sales = b0 + b1 * TV + error\n\nRun:  python tv_sales_linear_regression.py\nNeeds: pandas, numpy, matplotlib, seaborn, scipy, scikit-learn, statsmodels\n'

In [2]:
%matplotlib inline

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

OUT_DIR = "outputs_slr"
os.makedirs(OUT_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")


In [3]:

# ----------------------------------------------------------------------
# 1. Load data
#    (If your CSV has an extra index column, e.g. "Unnamed: 0", drop it.)
# ----------------------------------------------------------------------
path = next(p for p in ["advertising.csv", "/mnt/user-data/uploads/advertising.csv"]
            if os.path.exists(p))
df = pd.read_csv(path)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("=" * 60)
print("1. DATA")
print("=" * 60)
print("Shape:", df.shape)
print(df[["TV", "Sales"]].describe().round(2))
print("Missing values:", int(df[["TV", "Sales"]].isna().sum().sum()), "\n")

X = df[["TV"]]
y = df["Sales"]


1. DATA
Shape: (200, 4)
           TV   Sales
count  200.00  200.00
mean   147.04   15.13
std     85.85    5.28
min      0.70    1.60
25%     74.38   11.00
50%    149.75   16.00
75%    218.82   19.05
max    296.40   27.00
Missing values: 0 



In [4]:

# ----------------------------------------------------------------------
# 2. Is there a relationship? Correlation + scatter plot
# ----------------------------------------------------------------------
r, p_corr = stats.pearsonr(df["TV"], df["Sales"])
print("=" * 60)
print("2. CORRELATION")
print("=" * 60)
print(f"Pearson r = {r:.4f}  (p = {p_corr:.2e})\n")

plt.figure(figsize=(7, 5))
plt.scatter(df["TV"], df["Sales"], alpha=0.65)
plt.xlabel("TV advertising budget (thousands of $)")
plt.ylabel("Sales (thousands of units)")
plt.title(f"TV vs Sales (r = {r:.2f})")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_scatter.png", dpi=150)
plt.show()


2. CORRELATION
Pearson r = 0.9012  (p = 7.93e-74)



In [5]:

# ----------------------------------------------------------------------
# 3. Fit the simple linear regression (statsmodels OLS gives full inference)
# ----------------------------------------------------------------------
X_sm = sm.add_constant(df["TV"])          # adds the intercept column
ols = sm.OLS(y, X_sm).fit()

print("=" * 60)
print("3. OLS REGRESSION SUMMARY")
print("=" * 60)
print(ols.summary(), "\n")

b0, b1 = ols.params["const"], ols.params["TV"]
ci = ols.conf_int(alpha=0.05)
n = len(df)
rss = np.sum(ols.resid ** 2)
rse = np.sqrt(rss / (n - 2))              # residual standard error
print("=" * 60)
print("4. KEY RESULTS")
print("=" * 60)
print(f"Equation            : Sales = {b0:.4f} + {b1:.4f} x TV")
print(f"Intercept (b0)      : {b0:.4f}   95% CI [{ci.loc['const', 0]:.4f}, {ci.loc['const', 1]:.4f}]")
print(f"Slope (b1)          : {b1:.4f}   95% CI [{ci.loc['TV', 0]:.4f}, {ci.loc['TV', 1]:.4f}]")
print(f"Slope t-statistic   : {ols.tvalues['TV']:.2f}   p-value = {ols.pvalues['TV']:.2e}")
print(f"R-squared           : {ols.rsquared:.4f}")
print(f"Residual Std Error  : {rse:.4f}  ({rse / y.mean() * 100:.1f}% of mean Sales)")
print(f"F-statistic         : {ols.fvalue:.2f}   (p = {ols.f_pvalue:.2e})")
print()
print("Interpretation:")
print(f"  - Each extra $1,000 spent on TV is associated with about {b1 * 1000:.1f} more units sold.")
print(f"  - With no TV spend, predicted Sales are about {b0 * 1000:,.0f} units.")
print(f"  - TV spend explains {ols.rsquared * 100:.1f}% of the variation in Sales.\n")


3. OLS REGRESSION SUMMARY
                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.812
Model:                            OLS   Adj. R-squared:                  0.811
Method:                 Least Squares   F-statistic:                     856.2
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           7.93e-74
Time:                        17:11:57   Log-Likelihood:                -448.99
No. Observations:                 200   AIC:                             902.0
Df Residuals:                     198   BIC:                             908.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.9748     

In [6]:

# Regression line plot with confidence / prediction bands
x_grid = pd.DataFrame({"TV": np.linspace(df["TV"].min(), df["TV"].max(), 200)})
pred = ols.get_prediction(sm.add_constant(x_grid)).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5.5))
plt.scatter(df["TV"], df["Sales"], alpha=0.55, label="Observed")
plt.plot(x_grid["TV"], pred["mean"], color="red", lw=2, label="Fitted line")
plt.fill_between(x_grid["TV"], pred["mean_ci_lower"], pred["mean_ci_upper"],
                 color="red", alpha=0.25, label="95% CI of mean")
plt.fill_between(x_grid["TV"], pred["obs_ci_lower"], pred["obs_ci_upper"],
                 color="gray", alpha=0.15, label="95% prediction interval")
plt.xlabel("TV advertising budget (thousands of $)")
plt.ylabel("Sales (thousands of units)")
plt.title(f"Sales = {b0:.2f} + {b1:.4f} x TV")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_regression_line.png", dpi=150)
plt.show()


In [7]:

# ----------------------------------------------------------------------
# 5. Check the model assumptions (residual diagnostics)
# ----------------------------------------------------------------------
resid, fitted = ols.resid, ols.fittedvalues
shapiro_p = stats.shapiro(resid).pvalue
print("=" * 60)
print("5. DIAGNOSTICS")
print("=" * 60)
print(f"Shapiro-Wilk normality test of residuals: p = {shapiro_p:.4f}")
print(f"Durbin-Watson (independence, ~2 is good): {sm.stats.durbin_watson(resid):.3f}")
print("Note: the residual plot shows the spread growing with TV "
      "(mild heteroscedasticity) and a slight curve, so a straight line "
      "is a good but not perfect description.\n")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(fitted, resid, alpha=0.6)
axes[0].axhline(0, color="red", ls="--")
axes[0].set(xlabel="Fitted values", ylabel="Residuals", title="Residuals vs Fitted")
sm.qqplot(resid, line="s", ax=axes[1])
axes[1].set_title("Normal Q-Q plot")
sns.histplot(resid, kde=True, ax=axes[2])
axes[2].set(xlabel="Residual", title="Residual distribution")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_diagnostics.png", dpi=150)
plt.show()



5. DIAGNOSTICS
Shapiro-Wilk normality test of residuals: p = 0.5269
Durbin-Watson (independence, ~2 is good): 2.029
Note: the residual plot shows the spread growing with TV (mild heteroscedasticity) and a slight curve, so a straight line is a good but not perfect description.



In [8]:
# ----------------------------------------------------------------------
# 6. Out-of-sample check with scikit-learn (80/20 split)
# ----------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LinearRegression().fit(X_train, y_train)
y_hat = lr.predict(X_test)
print("=" * 60)
print("6. TRAIN/TEST EVALUATION (scikit-learn)")
print("=" * 60)
print(f"Intercept = {lr.intercept_:.4f}, Slope = {lr.coef_[0]:.4f}")
print(f"Train R2  = {r2_score(y_train, lr.predict(X_train)):.4f}")
print(f"Test  R2  = {r2_score(y_test, y_hat):.4f}")
print(f"Test RMSE = {np.sqrt(mean_squared_error(y_test, y_hat)):.4f}\n")



6. TRAIN/TEST EVALUATION (scikit-learn)
Intercept = 7.0071, Slope = 0.0555
Train R2  = 0.8135
Test  R2  = 0.8026
Test RMSE = 2.4700



In [9]:
# ----------------------------------------------------------------------
# 7. Predict Sales for new TV budgets
# ----------------------------------------------------------------------
new = pd.DataFrame({"TV": [50, 100, 150, 200, 250]})
new_pred = ols.get_prediction(sm.add_constant(new, has_constant="add")).summary_frame(alpha=0.05)
out = pd.concat([new, new_pred[["mean", "obs_ci_lower", "obs_ci_upper"]]], axis=1)
out.columns = ["TV", "Predicted Sales", "95% PI lower", "95% PI upper"]
print("=" * 60)
print("7. PREDICTIONS")
print("=" * 60)
print(out.round(2).to_string(index=False))
print(f"\nPlots saved in ./{OUT_DIR}/")

7. PREDICTIONS
 TV  Predicted Sales  95% PI lower  95% PI upper
 50             9.75          5.20         14.30
100            12.52          7.98         17.06
150            15.29         10.76         19.83
200            18.07         13.52         22.61
250            20.84         16.29         25.40

Plots saved in ./outputs_slr/
